# CFD Pipeline — Part 2: Monitoring & Results

Monitors submitted HPC jobs and retrieves results once simulations complete.

**Prerequisites:** Run `pipeline_dashboard.ipynb` (Part 1) first to generate cases,
mesh them, and submit them to HPC.

**Workflow:**
1. Scan `case_status.json` files for current pipeline state.
2. Refresh SLURM job statuses by polling the HPC scheduler.
3. Fetch results (timestep data, postProcessing, logs) once jobs complete.

**Resume-safe:** Close and reopen at any time. All state is read from `case_status.json`.

## 1. Configuration

Edit these paths before running the notebook.

In [ ]:
import os
import sys
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Root of the CFD-dataset repository (directory containing this notebook)
# Prefer an explicit override via CFD_REPO_ROOT; otherwise, search upwards
# from the current working directory for a folder that looks like the repo.
env_repo_root = os.getenv("CFD_REPO_ROOT")
if env_repo_root:
    REPO_ROOT = os.path.abspath(env_repo_root)
else:
    cwd = Path.cwd()
    candidate = cwd
    repo_root = None
    for parent in [candidate] + list(candidate.parents):
        if (parent / "configs").is_dir() and (parent / "Data").is_dir():
            repo_root = parent
            break
    if repo_root is None:
        repo_root = cwd
    REPO_ROOT = str(repo_root)

# Directory that contains the generated OpenFOAM case folders
CASES_OUTPUT_DIR = os.path.join(REPO_ROOT, "openFoamCases")

# OpenFOAM case template folder (used by taskmanager)
TEMPLATE_PATH = os.path.join(REPO_ROOT, "configs", "openfoam_template")

# Input data directory (downloads from generateInputs.py)
INPUT_DATA_DIR = os.path.join(REPO_ROOT, "Data", "downloads")

# Remote HPC path on Deucalion (used by taskManager for rsync/sbatch)
DEUCALION_PATH = "/projects/xxx-xxx-xxx-123"

# ── taskmanager config ───────────────────────────────────────────────────────
TASKMANAGER_CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "taskmanager_config.yaml")

print(f"REPO_ROOT        : {REPO_ROOT}")
print(f"CASES_OUTPUT_DIR : {CASES_OUTPUT_DIR}")
print(f"INPUT_DATA_DIR   : {INPUT_DATA_DIR}")
print(f"DEUCALION_PATH   : {DEUCALION_PATH}")

## 2. Imports and Initialisation

In [ ]:
import importlib
import json
from pathlib import Path
from datetime import datetime

import pandas as pd

try:
    import taskmanager
    # Force reload to pick up latest changes
    importlib.reload(taskmanager)
    from taskmanager import OpenFOAMCaseGenerator
    print("✓ taskmanager imported (with reload)")
except ImportError as e:
    print(f"✗ Could not import taskmanager: {e}")
    print("  Install the taskmanager package: pip install git+https://github.com/souravsud/taskManager.git")
    OpenFOAMCaseGenerator = None

# Initialise the case generator
if OpenFOAMCaseGenerator is not None:
    generator = OpenFOAMCaseGenerator(
        template_path=TEMPLATE_PATH,
        input_dir=INPUT_DATA_DIR,
        output_dir=CASES_OUTPUT_DIR,
        cluster_path=DEUCALION_PATH,
        config_path=TASKMANAGER_CONFIG_PATH,
    )
    print(f"✓ OpenFOAMCaseGenerator initialised with config: {generator.config_path}")
else:
    generator = None

## 3. Status Scanner

Reads every `case_status.json` found under `CASES_OUTPUT_DIR` and assembles a
pandas DataFrame.  **Re-run this cell at any time to refresh the view.**

In [ ]:
def derive_pipeline_status(status: dict) -> str:
    """
    Map the raw fields in case_status.json to a single human-readable
    pipeline status string:

        ready_to_mesh  – mesh has not been run yet (mesh_status == NOT_RUN)
        meshing        – meshing is currently in progress (mesh_status == IN_PROGRESS)
        meshed         – mesh done, not yet submitted to HPC
        running        – submitted to HPC, job is PENDING or RUNNING
        complete       – HPC job completed successfully
        failed         – meshing failed, or HPC job failed/cancelled/timed-out
        unknown        – status file present but unrecognised combination
    """
    mesh_status = (status.get("mesh_status") or "NOT_RUN").upper()
    job_status  = (status.get("job_status")  or "").upper()
    submitted   = status.get("submitted", False)

    if mesh_status in ("FAILED", "ERROR"):
        return "failed"
    if job_status in ("FAILED", "CANCELLED", "TIMEOUT"):
        return "failed"
    if job_status == "COMPLETED":
        return "complete"
    if submitted and job_status in ("PENDING", "RUNNING", ""):
        return "running"
    if mesh_status == "DONE":
        return "meshed"
    if mesh_status == "IN_PROGRESS":
        return "meshing"
    if mesh_status == "NOT_RUN":
        return "ready_to_mesh"
    return "unknown"


def scan_cases(output_dir: str) -> pd.DataFrame:
    """
    Walk *output_dir* and collect every case_status.json into a DataFrame.
    Returns an empty DataFrame (with the expected columns) if no cases exist yet.
    """
    output_path = Path(output_dir)
    records = []

    if not output_path.exists():
        print(f"⚠  Cases output directory does not exist yet: {output_dir}")
        print("   Run generateInputs.py and the taskManager case generator first.")
    else:
        for status_file in sorted(output_path.rglob("case_status.json")):
            case_dir = status_file.parent
            try:
                with open(status_file) as fh:
                    status = json.load(fh)
            except (json.JSONDecodeError, OSError) as exc:
                print(f"⚠  Could not read {status_file}: {exc}")
                continue

            records.append({
                "case_name"       : case_dir.name,
                "case_path"       : str(case_dir),
                "pipeline_status" : derive_pipeline_status(status),
                "mesh_status"     : status.get("mesh_status"),
                "mesh_ok"         : status.get("mesh_ok"),
                "copied_to_hpc"   : status.get("copied_to_hpc"),
                "submitted"       : status.get("submitted"),
                "job_id"          : status.get("job_id"),
                "job_status"      : status.get("job_status"),
                "last_checked"    : status.get("last_checked"),
            })

    columns = [
        "case_name", "case_path", "pipeline_status",
        "mesh_status", "mesh_ok", "copied_to_hpc",
        "submitted", "job_id", "job_status", "last_checked",
    ]
    df = pd.DataFrame(records, columns=columns) if records else pd.DataFrame(columns=columns)
    return df


# ── Scan ──────────────────────────────────────────────────────────────────────
df_cases = scan_cases(CASES_OUTPUT_DIR)
print(f"Scanned {len(df_cases)} case(s) at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 4. Summary Dashboard

Per-status breakdown and case tables. Re-run after each status refresh.

In [ ]:
# ── High-level counts ────────────────────────────────────────────────────────
STATUS_ORDER = ["ready_to_mesh", "meshing", "meshed", "running", "complete", "failed", "unknown"]

total = len(df_cases)
print(f"{'='*50}")
print(f"  CFD PIPELINE STATUS SUMMARY")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*50}")
print(f"  Total cases : {total}")
print()

if total > 0:
    counts = df_cases["pipeline_status"].value_counts()
    for status in STATUS_ORDER:
        n = counts.get(status, 0)
        if n > 0:
            icon = {"ready_to_mesh": "○", "meshing": "⟳", "meshed": "✓",
                    "running": "▶", "complete": "★", "failed": "✗"}.get(status, "?")
            print(f"  {icon}  {status:<16} : {n}")
print(f"{'='*50}")

In [ ]:
# ── Per-status counts as a styled DataFrame ───────────────────────────────────
if total > 0:
    count_df = (
        df_cases["pipeline_status"]
        .value_counts()
        .reindex(STATUS_ORDER)
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    count_df.columns = ["pipeline_status", "count"]
    count_df = count_df[count_df["count"] > 0]
    display(count_df.style.hide(axis="index").set_caption("Cases per pipeline status"))
else:
    print("No cases found.")

In [ ]:
# ── Ready-to-mesh cases ──────────────────────────────────────────────────────
df_ready = df_cases[df_cases["pipeline_status"] == "ready_to_mesh"][["case_name", "mesh_status"]]
print(f"Ready to mesh: {len(df_ready)} case(s)")
if not df_ready.empty:
    display(df_ready.reset_index(drop=True))

In [ ]:
# ── Running cases ────────────────────────────────────────────────────────────
df_running = df_cases[df_cases["pipeline_status"] == "running"][
    ["case_name", "job_id", "job_status", "last_checked"]
]
print(f"Running on HPC: {len(df_running)} case(s)")
if not df_running.empty:
    display(df_running.reset_index(drop=True))

In [ ]:
# ── Failed cases ─────────────────────────────────────────────────────────────
df_failed = df_cases[df_cases["pipeline_status"] == "failed"][
    ["case_name", "mesh_status", "job_id", "job_status"]
]
print(f"Failed cases: {len(df_failed)}")
if not df_failed.empty:
    display(df_failed.reset_index(drop=True))

In [ ]:
# ── Full case table ───────────────────────────────────────────────────────────
print("Full case table:")
if total > 0:
    display(
        df_cases[
            ["case_name", "pipeline_status", "mesh_status",
             "submitted", "job_id", "job_status", "last_checked"]
        ].reset_index(drop=True)
    )
else:
    print("No cases found. Run generateInputs.py and the taskmanager case generator first.")

## 5. Refresh Job Statuses

Polls the SLURM scheduler on Deucalion for every submitted case and updates the
local `case_status.json` files.  Requires SSH access to the `deucalion` host.

Re-run this cell periodically to check for completed jobs.

In [ ]:
if generator is None:
    print("✗ taskmanager not available.")
else:
    import traceback

    submitted_cases = generator.list_cases_by_status(submitted=True)
    print(f"Submitted cases to check: {len(submitted_cases)}")

    if not submitted_cases:
        print("No submitted jobs to refresh.")
        print("Possible reasons:")
        print("  • No cases have been submitted yet — run pipeline_dashboard.ipynb first.")
        print("  • All jobs have already been updated.")
    else:
        for case_path in submitted_cases:
            try:
                new_status = generator.update_job_status(case_path)
                print(f"  {Path(case_path).name}: {new_status}")
            except Exception as exc:
                print(f"  ✗ {Path(case_path).name}: {exc}")
                traceback.print_exc()

        print("\nJob statuses updated. Re-run Cell 3 to refresh the dashboard.")

## 6. Fetch Results from HPC

Once jobs have completed (`job_status == "COMPLETED"`), download the results back
to the local case directories.  This includes:
- The **last timestep directory** (e.g., `20000/`)
- **postProcessing/** folder (if it exists)
- **Log files** (`log.simpleFoam`, `log.decomposePar`, etc.)

This is resume-safe — already-fetched cases are skipped.

In [ ]:
if generator is None:
    print("✗ taskmanager not available.")
else:
    # Ensure generator is initialised even if Section 2 was not re-run
    if "generator" not in globals() or generator is None:
        generator = OpenFOAMCaseGenerator(
            template_path=TEMPLATE_PATH,
            input_dir=INPUT_DATA_DIR,
            output_dir=CASES_OUTPUT_DIR,
            cluster_path=DEUCALION_PATH,
            config_path=TASKMANAGER_CONFIG_PATH,
        )
        print(f"✓ OpenFOAMCaseGenerator initialised with config: {generator.config_path}")

    # Re-scan to pick up the latest status
    df_cases = scan_cases(CASES_OUTPUT_DIR)

    # Find completed (or failed) jobs that haven't had results fetched yet
    completed_or_failed = df_cases[
        df_cases["job_status"].fillna("").str.upper().isin(["COMPLETED", "FAILED", "TIMEOUT"])
    ]

    # Filter out those already fetched
    cases_to_fetch = []
    for _, row in completed_or_failed.iterrows():
        case_path = Path(row["case_path"])
        status = generator.get_status(case_path)

        if not status or not status.get("results_fetched", False):
            cases_to_fetch.append(case_path)

    if not cases_to_fetch:
        print("No completed jobs with unfetched results.")
        print("Either:")
        print("  • No jobs have completed yet — re-run Section 7 to refresh statuses")
        print("  • All results have already been fetched")
    else:
        print(f"Fetching results from {len(cases_to_fetch)} case(s)…\n")
        results = generator.fetch_multiple_results(
            cases_to_fetch,
            fetch_last_timestep=True,
            fetch_postprocessing=True,
            fetch_logs=True
        )

        succeeded = sum(results)
        failed = len(results) - succeeded
        print(f"\n✓ Results fetch complete: {succeeded} succeeded, {failed} failed.")
        print("Re-run Cell 3 to refresh the dashboard to see updated status.")